# 條件判斷與 `for` 迴圈：Python / C / C++

這份筆記整理三個容易混淆、但非常重要的程式設計觀念：

1. **巢狀 `if`（nested if）**：一個條件成立後，再檢查第二個條件。
2. **Python `for` 迴圈中的迴圈變數**：為什麼在迴圈內寫 `i += 1`，不會讓 `range()` 跳著走？
3. **C++ range-based for**：如何直接逐一取出陣列或容器中的元素。

---

## 學習目標

完成這份筆記後，你應該能夠：

- 看懂巢狀 `if` 的執行流程。
- 判斷某段條件程式在不同輸入下會輸出什麼。
- 理解 `for i in range(...)` 中 `i` 是如何取得下一個值的。
- 分辨 Python `for` 與 `while` 的控制方式。
- 使用 C++ 的 range-based `for` 逐一讀取陣列元素。
- 解釋 `for (int value : values)` 中每個部分的意思。


## 0. Notebook 環境設定

下面這個 cell 會載入課堂使用的 C / C++ Notebook 設定。

載入後可以使用：

```text
%%c
```

執行 C 程式，以及：

```text
%%cpp
```

執行 C++ 程式。

> 如果你的環境沒有 `C_setting.py`，這個 cell 可能無法執行；這不影響後面的觀念閱讀。


In [ ]:
from C_setting import *

print("C / C++ Notebook 環境已載入。")
print("可以開始使用 %%c 與 %%cpp。")


# 1. 巢狀 `if`：條件裡面再判斷條件

原始程式片段：

```c
if (has_ticket) {
    if (age >= 18) {
        printf("Enter\n");
    } else {
        printf("Too young\n");
    }
}
```

這種寫法叫做 **巢狀條件判斷（nested if）**。

程式不是一次檢查所有條件，而是分兩層：

```text
有票嗎？
│
├─ 否 → 什麼都不做
│
└─ 是
    │
    └─ 年齡 >= 18 嗎？
        ├─ 是 → Enter
        └─ 否 → Too young
```

因此第二個條件：

```c
age >= 18
```

只有在：

```c
has_ticket
```

成立時才會被檢查。


## 1.1 三種可能情況

| `has_ticket` | `age` | 執行結果 |
|---|---:|---|
| `false` | 20 | 沒有輸出 |
| `true` | 17 | `Too young` |
| `true` | 18 | `Enter` |
| `true` | 25 | `Enter` |

注意邊界：

```c
age >= 18
```

代表 **18 歲本身也符合條件**。

如果寫成：

```c
age > 18
```

那麼 18 歲就不符合。


## 1.2 可直接執行的 C 範例

原始片段沒有宣告 `has_ticket` 與 `age`，所以單獨執行會編譯失敗。

下面補上完整程式，方便測試。

你可以修改：

```c
int has_ticket = 1;
int age = 17;
```

觀察不同情況的結果。

在 C 中：

```text
0 → false
非 0 → true
```


In [ ]:
%%c

#include <stdio.h>

int main(void) {
    int has_ticket = 1;
    int age = 17;

    if (has_ticket) {
        if (age >= 18) {
            printf("Enter\n");
        } else {
            printf("Too young\n");
        }
    }

    return 0;
}


## 1.3 可以合併成一個條件嗎？

如果我們只關心「可不可以進入」，也可以使用 `&&`：

```c
if (has_ticket && age >= 18) {
    printf("Enter\n");
}
```

其中：

```text
&&
```

表示 **AND（而且）**。

也就是：

```text
有票 AND 年齡至少 18
```

兩個條件都成立才會進入。

不過，原本的巢狀 `if` 有一個優點：  
它可以針對不同失敗原因做不同處理，例如：

```text
沒票 → No ticket
有票但未滿 18 → Too young
```


## 1.4 常見錯誤

### 錯誤 1：把比較寫成指定

正確：

```c
if (age == 18)
```

錯誤：

```c
if (age = 18)
```

`=` 是指定值，`==` 才是比較。

---

### 錯誤 2：忘記大括號所屬範圍

縮排雖然能幫助閱讀，但 C / C++ 真正判斷程式區塊的是：

```text
{ }
```

---

### 錯誤 3：忽略邊界值

```c
age >= 18
```

和：

```c
age > 18
```

結果不同。

寫條件時一定要特別測試：

```text
17
18
19
```


# 2. Python `for`：為什麼 `i += 1` 不會讓迴圈跳著走？

原始程式：

```python
for i in range(10):
    # <...>
    i += 1
```

很多初學者會以為：

```python
i += 1
```

會讓下一輪的 `i` 再多增加 1。

例如以為會變成：

```text
0, 2, 4, 6, 8
```

但實際上 **不會**。

`range(10)` 會依序提供：

```text
0, 1, 2, 3, 4, 5, 6, 7, 8, 9
```

每次進入下一輪迴圈時，Python 會重新把 `range(10)` 提供的下一個值指定給 `i`。


## 2.1 逐輪追蹤

假設程式是：

```python
for i in range(3):
    print("before:", i)
    i += 1
    print("after: ", i)
```

執行流程：

| 第幾輪 | `range()` 給 `i` 的值 | `i += 1` 後 |
|---:|---:|---:|
| 1 | 0 | 1 |
| 2 | 1 | 2 |
| 3 | 2 | 3 |

最重要的是第二輪開始時：

```text
上一輪最後 i = 1
```

並不是 Python 自己再加 1。

而是：

```text
range() 提供下一個值 1
↓
Python 將 i 重新指定成 1
```

因此修改 `i` **不會改變 `range()` 本身的序列**。


In [ ]:
for i in range(3):
    print("before:", i)
    i += 1
    print("after: ", i)
    print("---")


## 2.2 如果真的想一次增加 2 呢？

不要在迴圈裡修改 `i`，而是直接修改 `range()`：

```python
for i in range(0, 10, 2):
    print(i)
```

其中：

```python
range(start, stop, step)
```

代表：

```text
start = 起點
stop  = 終點（不包含）
step  = 每次增加多少
```

因此：

```python
range(0, 10, 2)
```

產生：

```text
0, 2, 4, 6, 8
```


In [ ]:
for i in range(0, 10, 2):
    print(i)


## 2.3 `for` 和 `while` 的差異

如果「下一次的值」要由你自己控制，`while` 通常更直覺。

### `for`

```python
for i in range(5):
    print(i)
```

Python 負責依序提供下一個值。

### `while`

```python
i = 0

while i < 5:
    print(i)
    i += 1
```

這裡的：

```python
i += 1
```

非常重要。

如果忘記它：

```python
i = 0

while i < 5:
    print(i)
```

`i` 永遠是 0，條件永遠成立，就會形成 **無窮迴圈（infinite loop）**。


## 2.4 小測驗

先不要執行，猜猜看輸出：

```python
for i in range(4):
    i += 10
    print(i)
```

你預測的是：

```text
?
?
?
?
```

### 思考方式

`range(4)` 依序提供：

```text
0, 1, 2, 3
```

每一輪再各自加 10。

因此答案是：

```text
10
11
12
13
```


In [ ]:
for i in range(4):
    i += 10
    print(i)


# 3. C++ Range-based `for`

原始程式：

```cpp
int values[] = {10, 20, 30};

for (int value : values) {
    std::cout << value << '\n';
}
```

這是 C++11 開始提供的 **range-based for loop**。

用途是：

> 對陣列或容器裡的每一個元素，重複執行一次程式。


## 3.1 語法拆解

```cpp
for (int value : values)
```

可以拆成：

```text
for (
    int value     :     values
)
    ↑                    ↑
每次取出的元素       要走訪的資料集合
```

如果：

```cpp
int values[] = {10, 20, 30};
```

那麼每一輪：

```text
第 1 輪 → value = 10
第 2 輪 → value = 20
第 3 輪 → value = 30
```

因此輸出：

```text
10
20
30
```


In [ ]:
%%cpp

#include <iostream>

int main() {
    int values[] = {10, 20, 30};

    for (int value : values) {
        std::cout << value << '\n';
    }

    return 0;
}


## 3.2 和傳統索引 `for` 比較

傳統寫法：

```cpp
for (int i = 0; i < 3; ++i) {
    std::cout << values[i] << '\n';
}
```

range-based 寫法：

```cpp
for (int value : values) {
    std::cout << value << '\n';
}
```

如果你只是想：

```text
把每個元素都讀一次
```

range-based `for` 通常更簡潔。

但如果你需要：

- 元素的索引 `i`
- 只走訪某一部分
- 每隔幾個元素取一次

傳統索引 `for` 可能更適合。


## 3.3 `value` 是複製品

這個寫法：

```cpp
for (int value : values)
```

每一輪會把元素的值 **複製** 到 `value`。

所以：

```cpp
value += 100;
```

只會改到 `value`，不會修改原本陣列。

例如：

```cpp
int values[] = {10, 20, 30};

for (int value : values) {
    value += 100;
}
```

最後 `values` 仍然是：

```text
10 20 30
```


In [ ]:
%%cpp

#include <iostream>

int main() {
    int values[] = {10, 20, 30};

    for (int value : values) {
        value += 100;
    }

    for (int value : values) {
        std::cout << value << ' ';
    }

    std::cout << '\n';

    return 0;
}


## 3.4 如果想修改原本元素：使用 reference

可以寫成：

```cpp
for (int& value : values)
```

`&` 表示 `value` 參考到原本的元素，而不是建立一份獨立複製品。

因此：

```cpp
for (int& value : values) {
    value += 100;
}
```

會真的把陣列改成：

```text
110 120 130
```

目前先記住：

```text
int value
→ 取得一份值的複製

int& value
→ 直接操作原本元素
```


In [ ]:
%%cpp

#include <iostream>

int main() {
    int values[] = {10, 20, 30};

    for (int& value : values) {
        value += 100;
    }

    for (int value : values) {
        std::cout << value << ' ';
    }

    std::cout << '\n';

    return 0;
}


## 3.5 只讀取、不修改：`const`

如果只想讀取元素，而且不希望不小心修改，可以寫：

```cpp
for (const int& value : values)
```

這表示：

```text
const → 不允許透過 value 修改元素
&     → 不需要建立額外複製
```

對 `int` 這種很小的型別，直接複製通常沒有問題。

但未來遇到：

```cpp
std::string
大型物件
自訂 class
```

常會看到：

```cpp
const T&
```

這種形式。


# 4. Python `for` 與 C++ range-based `for` 的共同概念

Python：

```python
for value in values:
    print(value)
```

C++：

```cpp
for (int value : values) {
    std::cout << value << '\n';
}
```

雖然語法不同，但想法非常接近：

```text
從集合中取出一個元素
        ↓
執行迴圈內容
        ↓
取得下一個元素
        ↓
重複直到沒有元素
```

這種寫法的重點是：

> 我們關心「每個元素」，而不一定關心它是第幾個元素。


# 5. 常見錯誤整理

## Python

### ❌ 以為修改 `i` 可以控制下一輪

```python
for i in range(10):
    i += 1
```

下一輪的值仍然由 `range(10)` 決定。

若要控制間距：

```python
range(0, 10, 2)
```

---

## C++

### ❌ 在 Python cell 直接寫 C++ range-based for

```cpp
for (int value : values) {
}
```

Python 不認得這種語法。

在這份 Notebook 中要使用：

```text
%%cpp
```

---

### ❌ 忘記變數或陣列的型別

C++ 需要：

```cpp
int values[] = {10, 20, 30};
```

而不是只寫：

```cpp
values = {10, 20, 30};
```

---

### ❌ 想修改陣列，卻使用值複製

```cpp
for (int value : values)
```

不能透過 `value` 修改原本元素。

需要：

```cpp
for (int& value : values)
```
